# 02 – Preprocessing & Data Cleaning

### Purpose of the Notebook
This notebook applies systematic cleaning and standardisation to the pre‑saved datasets (dataset.pkl and dataset_de.pkl).
All decisions are based on the insights from Notebook 01_data_overview (EDA), including handling of missing data, removal of low‑quality fields, type corrections, logical consistency checks, and creation of derived features.

### Steps
- Load pre‑saved datasets
- Apply preprocessing pypline
- Save cleaned datasets

--------------------
#### Imports & Setup & Dataset
-------------------

In [15]:
# ---------------------------------------------------------
# Import moduls
# ---------------------------------------------------------


# import standard modules
import pandas as pd
import numpy as np
from pathlib import Path

import sys
from pathlib import Path

In [16]:
# shut off some annoying warnings
import warnings

warnings.filterwarnings("ignore", message="A value is trying to be set on a copy")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [17]:
# ---------------------------------------------------------
# Setup style
# ---------------------------------------------------------

# show all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# visualisation settings
pd.set_option('display.float_format', '{:,.2f}'.format)

In [18]:
# ---------------------------------------------------------
# Load scripts
# ----------------------------------------------------------

%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

# found min directory
PROJECT_ROOT = Path("..").resolve()

# maindirectory sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# import function from script
from my_scripts.preprocessing import preprocess
from my_scripts.eda import overview

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:
# ---------------------------------------------------------
# Import dataset
# ---------------------------------------------------------

df = pd.read_pickle("../data/dataset.pkl")

print("EU dataset:", df.shape)

EU dataset: (4362869, 75)


--------------
### Apply preprocessing pipeline
-----------

In [20]:
# apply funktion
df_clean = preprocess(df)


In [21]:
# shape of the cleaned dataset
print(df_clean.shape)

(2993666, 14)


In [22]:
df_clean.head()

,YEAR,ISO_COUNTRY_CODE,CAE_TYPE,TYPE_OF_CONTRACT,TOP_TYPE,CPV,MAIN_CPV_CODE_GPA,AWARD_VALUE_EURO,CRIT_PRICE_WEIGHT,NUMBER_OFFERS,DT_DISPATCH,DT_AWARD,TITLE,CRIT_WEIGHTS
0,2008,DE,8,W,OPE,45215000.0,Unknown,"302,964.15",NaN,2.00,2007-12-11,2007-09-29,Unknown,60---40
1,2008,DE,3,W,OPE,45215100.0,Unknown,"478,780.08",100.00,3.00,2007-12-27,2007-12-21,Unknown,Unknown
2,2008,ES,1,W,OPE,45234116.0,Unknown,"14,237,368.10",NaN,6.00,2007-12-27,2007-10-25,Unknown,60---40
3,2008,DE,3,W,OPE,45233110.0,Unknown,"5,204,457.56",NaN,9.00,2007-12-27,2007-11-30,Unknown,90---10
4,2008,ES,3,W,OPE,45233222.0,Unknown,"4,924,202.16",NaN,45.00,2007-12-28,2007-12-14,Unknown,70---30


In [23]:
df_clean.describe().T

,count,mean,min,25%,50%,75%,max,std
YEAR,"2,993,666.00","2,012.35","2,008.00","2,010.00","2,012.00","2,015.00","2,016.00",2.53
AWARD_VALUE_EURO,"2,993,666.00","6,248,001.75",0.01,"6,409.16","64,251.17","413,596.25","9,999,999,999.00","91,502,584.14"
CRIT_PRICE_WEIGHT,"1,470,165.00",100.34,0.00,100.00,100.00,100.00,"1,487,600.00","1,289.20"
NUMBER_OFFERS,"2,993,666.00",7.03,0.00,1.00,3.00,6.00,500.00,18.61
DT_DISPATCH,2993666,2012-11-02 16:38:16.859034,2007-07-16 00:00:00,2010-10-04 00:00:00,2012-12-19 00:00:00,2015-01-21 00:00:00,2016-12-30 00:00:00,NaN
DT_AWARD,2958515,2012-09-14 13:28:12.071731,1996-09-25 00:00:00,2010-08-09 00:00:00,2012-11-05 00:00:00,2014-12-09 00:00:00,2028-10-04 00:00:00,NaN


#### Notes: Summary of Data Cleaning Results

1. Dataset Size After Preprocessing
- Before: 4,039,906 rows × 75 columns
- After: 2,993,666 rows × 14 columns
The reduction reflects removal of non-informative fields, consolidation of text columns, elimination of corrupted numeric values, and reconstruction of award amounts from multiple financial fields.

2. Columns Removed During Preprocessing

Removed due to >40% missing values or structural corruption:
- VALUE_EURO_FIN_1, VALUE_EURO_FIN_2
- AWARD_VALUE_EURO_FIN_1
- AWARD_EST_VALUE_EURO
- Winner information (WIN_*)
- Contracting authority details (CAE_*)
- GPA-related fields
- Award criteria weights (CRIT_*, except CRIT_PRICE_WEIGHT)
- Additional CPVs
- Procedural flags with extreme cardinality (B_MULTIPLE_, B_FRA_, FRA_ESTIMATED, etc.)
- TED_NOTICE_URL

Removed due to irrelevance for competition modelling:
- Identifiers (ID_NOTICE_CAN, ID_AWARD, ID_LOT_AWARDED, CONTRACT_NUMBER)
- Non-award information (INFO_ON_NON_AWARD, INFO_UNPUBLISHED)
- Administrative metadata (MAIN_ACTIVITY, EU_INST_CODE)

Removed due to consolidation:
- TITLE (merged into a single NLP field used for TF-IDF/SVD/NMF/SVM)

3. Reconstruction and Cleaning of Award Values

3.1. Multi-source reconstruction of AWARD_VALUE_EURO  
Missing award values were filled using any realistic numeric value (≤10 billion EUR) found in:
- VALUE_EURO
- VALUE_EURO_FIN_1
- VALUE_EURO_FIN_2
- AWARD_EST_VALUE_EURO
- AWARD_VALUE_EURO_FIN_1  
This step reduced missing award values from ~60% to <15%.

3.2. Removal of unrealistic values  
All values above 10,000,000,000 EUR were removed as technical artifacts.

3.3. Final removal of remaining missing award values  
After reconstruction, remaining missing AWARD_VALUE_EURO (~15%) were dropped, as tenders without any valid financial information cannot be used for competition or failure-risk modelling.

4. Additional Preprocessing Steps
- CANCELLED tenders removed entirely.
- All categorical missing values replaced with "Unknown".
- All categorical variables converted to string to ensure stable feature engineering.
- Numeric columns converted to float with coercion to handle corrupted formats.
- Date fields parsed into datetime.
- High-cardinality categorical fields removed to prevent memory issues and unstable encoding.


--------------
### Save cleaned datasets

-----------

In [24]:
df_clean.to_pickle("../data/dataset_clean.pkl")